# Modelo v3 do MAPi — de ponta a ponta

**DADOS SINTÉTICOS — experimento controlado.** O rótulo vem da fórmula do v2 (`reports/synthetic_ground_truth.md`); nenhum número deste caderno é desempenho em alagamento real.

DADOS DA REDE OBSERVACIONAL DO CEMADEN/MCTIC.

Este caderno regenera `reports/v3/` e `models/v3/` a partir dos arquivos de `data/in1165/` e `data/v3/` (sem banco). O v2 (`reports/`, `models/`) não é tocado.

| Etapa | O que faz | Módulo |
|---|---|---|
| B1, B2, B3 | Série contínua com alvo de 0, 3 e 6 h, lacunas do padrão real, janelas disjuntas | `mapi_ai.v3.dataset` |
| B4 | Quatro candidatos; escolha na validação, critério gravado antes | `mapi_ai.v3.train` |
| B6, B7 | Calibração, correção de prevalência, limiares por orçamento, eventos | `mapi_ai.v3.evaluate` |
| B5 | Variante com entradas físicas (gerador próprio) | `mapi_ai.v3.physical` |
| B8 | Explicações: acumulados aninhados × janelas disjuntas | `mapi_ai.v3.explain` |

In [1]:
import os, sys
from pathlib import Path
# O caderno roda de notebooks/: a raiz do repositório entra no caminho e vira o diretório de trabalho.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import json, time
import pandas as pd
from mapi_ai.v3 import config as C
from mapi_ai.v3 import dataset as D, train as T, evaluate as E, explain as X, physical as P, serve as S, model_card as MC
t0 = time.time()

## 1. Dataset (B1, B2, B3)

O rótulo é gerado sobre a chuva **verdadeira**; as lacunas entram **depois**, só na chuva que o modelo vê.

In [2]:
summary = D.write()
pd.DataFrame({'prevalência na série contínua': summary['continuous']['prevalence_continuous']})

,prevalência na série contínua
h0,0.032966
h3,0.086436
h6,0.130696


In [3]:
pd.DataFrame(summary['sample']['split']).T[['rows', 'positives_h0', 'positives_h3', 'positives_h6', 'from', 'to']]

,rows,positives_h0,positives_h3,positives_h6,from,to
train,3017,290,610,804,2021-10-01 20:00:00+00:00,2024-07-03 06:00:00+00:00
validation,965,67,123,184,2024-07-03 23:00:00+00:00,2025-07-30 13:00:00+00:00
test,1018,63,143,201,2025-07-31 02:00:00+00:00,2026-10-06 12:00:00+00:00


In [4]:
pd.DataFrame(summary['continuous']['gaps']['by_point']).T

,alvo_fracao_curta,obtida,horas
AGAMENON_DERBY,0.00239,0.00257,115.0
AV_RECIFE_IBURA,0.00775,0.00778,348.0
CIN_UFPE,0.00388,0.00403,180.0
JABOATAO_CENTRO,0.00180,0.00206,92.0
MASCARENHAS_IMBIRIBEIRA,0.00291,0.00315,141.0


## 2. Treino e escolha do modelo (B4)

Critério fixado antes de treinar: maior PR-AUC na validação; empate dentro do IC 95% do bootstrap pareado fica com o mais simples.

In [5]:
meta = T.run()
pd.DataFrame([{**r, 'horizonte': h} for h, v in meta['horizons'].items() for r in v['selection']['table']])

H=0: melhor=logistica escolhido=logistica regra_chuva3h=0.725 logistica=0.790 arvore_rasa=0.626 xgboost=0.771


H=3: melhor=logistica escolhido=logistica regra_chuva3h=0.654 logistica=0.729 arvore_rasa=0.550 xgboost=0.720


H=6: melhor=logistica escolhido=logistica regra_chuva3h=0.601 logistica=0.672 arvore_rasa=0.541 xgboost=0.665


,model,val_pr_auc,diff_best_minus_model,ci95_lo,ci95_hi,tied_with_best,horizonte
0,regra_chuva3h,0.725329,0.064504,0.028535,0.107579,False,h0
1,logistica,0.789833,0.000000,0.000000,0.000000,True,h0
2,arvore_rasa,0.625876,0.163957,0.093897,0.237081,False,h0
3,xgboost,0.770799,0.019034,-0.010206,0.053316,True,h0
4,regra_chuva3h,0.654006,0.074742,0.039329,0.110584,False,h3
5,logistica,0.728749,0.000000,0.000000,0.000000,True,h3
6,arvore_rasa,0.549868,0.178880,0.135305,0.223098,False,h3
7,xgboost,0.719705,0.009044,-0.009324,0.028094,True,h3
8,regra_chuva3h,0.601391,0.070765,0.039996,0.101314,False,h6
9,logistica,0.672156,0.000000,0.000000,0.000000,True,h6


## 3. Calibração, prevalência, limiares e eventos (B6, B7)

In [6]:
S.write_background()
ev = E.run()
pd.read_csv(C.REPORTS_DIR / 'metrics.csv').query("part == 'test'")

,generation,horizon_h,model,part,inputs,probability,n,positives,pr_auc,roc_auc,brier,bss,ece
1,v3,0,regra_chuva3h,test,com lacunas,bruta,1018,63,0.659284,0.868736,0.039651,0.330543,0.029825
2,v3,0,regra_chuva3h,test,sem lacunas,bruta,1018,63,0.665865,0.878783,0.039484,0.333364,0.029673
4,v3,0,logistica,test,com lacunas,bruta,1018,63,0.677355,0.872600,0.047062,0.205406,0.089191
5,v3,0,logistica,test,sem lacunas,bruta,1018,63,0.726855,0.911144,0.045028,0.239752,0.095830
7,v3,0,arvore_rasa,test,com lacunas,bruta,1018,63,0.551367,0.854251,0.055858,0.056901,0.120624
8,v3,0,arvore_rasa,test,sem lacunas,bruta,1018,63,0.551367,0.854251,0.055858,0.056901,0.120624
10,v3,0,xgboost,test,com lacunas,bruta,1018,63,0.697875,0.898321,0.047092,0.204903,0.102024
11,v3,0,xgboost,test,sem lacunas,bruta,1018,63,0.708873,0.904621,0.045829,0.226228,0.099077
13,v3,0,logistica,test,com lacunas,calibrada,1018,63,0.677355,0.872600,0.031381,0.470170,0.015346
15,v3,3,regra_chuva3h,test,com lacunas,bruta,1018,143,0.574822,0.766561,0.109571,0.120252,0.108410


In [7]:
rows = []
for hk, v in ev['horizons'].items():
    for lvl in ('medium', 'high'):
        t = v['alerts_and_events']['test'][lvl]
        rows.append({'horizonte': hk, 'nível': lvl, 'limiar': v['thresholds'][lvl], 'dias de alerta por ponto por ano': t['alert_days_per_point_year'],
                     'eventos detectados': t['event_recall'], 'eventos em chuva detectados': t['rain_event_recall'],
                     'antecedência mediana (h)': t['lead_median_h'], 'falsos por evento': t['false_alerts_per_event']})
pd.DataFrame(rows)

,horizonte,nível,limiar,dias de alerta por ponto por ano,eventos detectados,eventos em chuva detectados,antecedência mediana (h),falsos por evento
0,h0,medium,0.004993,47.744304,0.296422,0.459103,0.0,0.122658
1,h0,high,0.870412,9.111508,0.083475,0.129288,0.0,0.000000
2,h3,medium,0.026667,50.295526,0.323680,0.501319,0.0,0.109029
3,h3,high,0.948358,9.111508,0.083475,0.129288,0.0,0.000000
4,h6,medium,0.049075,49.931066,0.332198,0.511873,0.0,0.098807
5,h6,high,0.927913,8.929278,0.081772,0.126649,0.0,0.000000


**Antecedência de 0 h.** Com chuva e maré só da hora `t`, o primeiro alerta sai quando o evento começa. Isso não muda com o horizonte do alvo.

## 4. Entradas físicas (B5)

O gerador muda nesta variante; a comparação é entre conjuntos de entradas **no mesmo rótulo novo**.

In [8]:
phys = P.run()
pd.DataFrame({k: {'PR-AUC teste': v['test_pr_auc_chosen'], 'sem solo, como em produção': v.get('test_pr_auc_soil_absent_as_in_production')} for k, v in phys['variants'].items()}).T

H=6: melhor=xgboost escolhido=logistica regra_chuva3h=0.655 logistica=0.687 arvore_rasa=0.569 xgboost=0.688


fisico_base logistica {'regra_chuva3h': 0.655, 'logistica': 0.687, 'arvore_rasa': 0.569, 'xgboost': 0.688} teste 0.647 None


H=6: melhor=xgboost escolhido=logistica regra_chuva3h=0.655 logistica=0.692 arvore_rasa=0.571 xgboost=0.696


fisico logistica {'regra_chuva3h': 0.655, 'logistica': 0.692, 'arvore_rasa': 0.571, 'xgboost': 0.696} teste 0.642 0.644753386747631


H=6: melhor=xgboost escolhido=logistica regra_chuva3h=0.655 logistica=0.684 arvore_rasa=0.569 xgboost=0.689


fisico_sem_solo logistica {'regra_chuva3h': 0.655, 'logistica': 0.684, 'arvore_rasa': 0.569, 'xgboost': 0.689} teste 0.647 None


,PR-AUC teste,"sem solo, como em produção"
fisico_base,0.647067,NaN
fisico,0.642305,0.644753
fisico_sem_solo,0.647292,NaN


## 5. Explicações (B8)

Mesma logística, mesmo dataset; só as entradas de chuva mudam.

In [9]:
xai = X.run()
pd.DataFrame({k: {'Jaccard SHAP × LIME': v['jaccard_top5_shap_lime'], 'Spearman SHAP × LIME': v['spearman_shap_lime'],
                  'estabilidade do LIME': v['lime_stability_10_seeds'], 'deleção SHAP k=1': v['deletion_k1']['SHAP'],
                  'deleção aleatória k=1': v['deletion_k1']['aleatório']} for k, v in xai['v3'].items()}).T

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


H=0 acumulados_aninhados: Jaccard 0.77 Spearman 0.77 LIME estável 0.80 deleção SHAP/LIME/acaso 0.046/0.033/0.012 fórmula 0.87


Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


H=0 janelas_disjuntas: Jaccard 0.62 Spearman 0.68 LIME estável 0.75 deleção SHAP/LIME/acaso 0.051/0.035/0.011 fórmula 0.53


Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


H=6 acumulados_aninhados: Jaccard 0.84 Spearman 0.81 LIME estável 0.89 deleção SHAP/LIME/acaso 0.015/0.004/0.008


H=6 janelas_disjuntas: Jaccard 0.70 Spearman 0.74 LIME estável 0.87 deleção SHAP/LIME/acaso 0.028/0.021/0.009


,Jaccard SHAP × LIME,Spearman SHAP × LIME,estabilidade do LIME,deleção SHAP k=1,deleção aleatória k=1
h0_acumulados_aninhados,0.770476,0.768970,0.799063,0.045803,0.011712
h0_janelas_disjuntas,0.625000,0.682909,0.750381,0.050618,0.011485
h6_acumulados_aninhados,0.839048,0.813091,0.887661,0.014639,0.007768
h6_janelas_disjuntas,0.697143,0.739152,0.867090,0.027787,0.008843


## 6. Cartão do modelo

In [10]:
card = MC.build()
(C.REPORTS_DIR / 'model_card.json').write_text(json.dumps(card, indent=2, ensure_ascii=False))
print(f"pronto em {time.time() - t0:.0f} s; modelo de produção: {card['modelVersion']} ({card['model']}), horizonte {card['horizonHours']} h")

pronto em 1132 s; modelo de produção: logreg_v3 (logistica), horizonte 6 h
